# BlackPearl Morphix — Stage 2, Step 1: Explore the Perturb-FISH tumour data

**Goal:** find out exactly what is in the public Perturb-FISH tumour dataset (Binan et al., 2025, *Cell*) before designing the Stage 2 model.

In that study, human melanoma cells (A375) carrying CRISPR knockouts of ~35 genes were grown as tumours in mice together with human T cells, then imaged with MERFISH. So each cell should have a **position**, a **cell type** and a **knockout**, which is the paired perturbation → tissue data Stage 2 needs.

**This notebook only downloads and inspects the data. No training.** A normal CPU runtime is fine. It takes 2–5 minutes.

At the end, copy the whole **SUMMARY** block and paste it back.

Data: Brain Image Library, `extras/tumors/processed/finaltables/` (linked from https://sspsygene.ucsc.edu/data).

In [ ]:
import os, io, json, urllib.request
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

BASE = 'https://download.brainimagelibrary.org/0c/bd/0cbd479c521afff9/extras/tumors/processed/finaltables/'
FILES = ['coordinates.csv', 'allcellsPerturbationTable.csv', 'tumorpooledperturbations.csv',
         'perturbed_tcells_perturbation.csv', 'withneighborperturbation.csv',
         'tumorMerfish.csv', 'perturbed_tcells_merfish.csv']
os.makedirs('pfish', exist_ok=True)

def fetch(name):
    path = os.path.join('pfish', name)
    if not os.path.exists(path):
        req = urllib.request.Request(BASE + name, headers={'User-Agent': 'Mozilla/5.0'})
        with urllib.request.urlopen(req, timeout=300) as r, open(path, 'wb') as f:
            f.write(r.read())
    return path

for f in FILES:
    try:
        p = fetch(f); print(f'OK   {f:40s} {os.path.getsize(p)/1e6:7.1f} MB')
    except Exception as e:
        print(f'FAIL {f:40s} {e}')

## Look inside every table

In [ ]:
def sniff(path, nrows=2000):
    # read a sample; detect whether the first column is an index
    df = pd.read_csv(path, nrows=nrows)
    if df.columns[0].startswith('Unnamed') or df.columns[0] == '':
        df = pd.read_csv(path, nrows=nrows, index_col=0)
    return df

info = {}
for f in FILES:
    p = os.path.join('pfish', f)
    if not os.path.exists(p):
        continue
    df = sniff(p)
    with open(p, 'rb') as fh:
        n_lines = sum(1 for _ in fh) - 1
    num = df.select_dtypes('number')
    binary = bool(len(num.columns)) and bool(np.isin(num.values[~np.isnan(num.values)], [0, 1]).all())
    info[f] = dict(rows=n_lines, cols=df.shape[1], first_cols=[str(c) for c in df.columns[:12]],
                   last_cols=[str(c) for c in df.columns[-5:]], index_example=[str(i) for i in df.index[:3]],
                   numeric_all_binary=binary)
    print(f'\n=== {f}: ~{n_lines} rows x {df.shape[1]} cols | numeric values all 0/1: {binary}')
    print('index examples:', list(df.index[:3]))
    with pd.option_context('display.max_columns', 12, 'display.width', 200):
        print(df.iloc[:3, :12])

## Plot the tissue (coordinates)

In [ ]:
coords = sniff(os.path.join('pfish', 'coordinates.csv'), nrows=None)
num_cols = coords.select_dtypes('number').columns.tolist()
print('numeric columns in coordinates.csv:', num_cols)
xcol = next((c for c in num_cols if 'x' in str(c).lower()), num_cols[0])
ycol = next((c for c in num_cols if 'y' in str(c).lower() and c != xcol), num_cols[1])
print('using', xcol, ycol)
cat_cols = [c for c in coords.columns if coords[c].dtype == object and 1 < coords[c].nunique() < 50]
print('categorical columns (possible sample / cell type):', {c: coords[c].nunique() for c in cat_cols})

fig, ax = plt.subplots(figsize=(8, 8))
if cat_cols:
    for g, d in coords.groupby(cat_cols[0]):
        ax.scatter(d[xcol], d[ycol], s=0.5, label=str(g))
    ax.legend(markerscale=10, fontsize=7)
else:
    ax.scatter(coords[xcol], coords[ycol], s=0.5)
ax.set_aspect('equal'); ax.set_title('Perturb-FISH tumour: cell positions')
plt.show()

d, _ = __import__('scipy.spatial', fromlist=['cKDTree']).cKDTree(coords[[xcol, ycol]].values).query(coords[[xcol, ycol]].values, k=2)
info['coordinates_extra'] = dict(n_cells=len(coords), xcol=str(xcol), ycol=str(ycol),
                                 cat_cols={c: int(coords[c].nunique()) for c in cat_cols},
                                 median_nn_distance=float(np.median(d[:, 1])),
                                 x_range=[float(coords[xcol].min()), float(coords[xcol].max())],
                                 y_range=[float(coords[ycol].min()), float(coords[ycol].max())])

## Perturbation assignments

In [ ]:
pert = sniff(os.path.join('pfish', 'allcellsPerturbationTable.csv'), nrows=None)
num = pert.select_dtypes('number')
print('shape:', pert.shape, '| numeric cols:', num.shape[1])
summary_p = dict(shape=list(pert.shape))
if num.shape[1] > 5:
    # wide matrix: cells x guides (or guides x cells)
    per_cell = (num > 0).sum(1)
    print('perturbations detected per row (cell?):', per_cell.value_counts().sort_index().head(10).to_dict())
    per_guide = (num > 0).sum(0).sort_values(ascending=False)
    print('rows per column (cells per guide?), top 40:'); print(per_guide.head(40).to_string())
    summary_p.update(per_row_counts=per_cell.value_counts().sort_index().head(10).to_dict(),
                     per_col_top=per_guide.head(40).to_dict(), n_cols_with_any=int((per_guide > 0).sum()))
else:
    print(pert.head(10))
    for c in pert.columns:
        if pert[c].dtype == object:
            vc = pert[c].value_counts()
            print(f'{c}: {pert[c].nunique()} unique; top:', vc.head(40).to_dict())
            summary_p[f'{c}_top'] = vc.head(40).to_dict()
info['perturbation_extra'] = summary_p

# do the coordinates and perturbation tables share cell IDs?
overlap = len(set(map(str, coords.index)) & set(map(str, pert.index)))
info['id_overlap_coords_vs_pert_index'] = overlap
print('cell IDs shared between coordinates and perturbation table (by index):', overlap)

## SUMMARY — copy everything printed below and paste it back

In [ ]:
print('===== SUMMARY START =====')
print(json.dumps(info, indent=1, default=str)[:12000])
print('===== SUMMARY END =====')